# card_stack

> Card stack operations for the alignment column: navigation, viewport update, width save

In [ ]:
#| default_exp routes.alignment.card_stack

In [ ]:
#| export
from typing import Any, Dict, List, Tuple

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.core.constants import DEFAULT_CARD_WIDTH
from cjm_fasthtml_card_stack.routes.handlers import (
    build_nav_response, card_stack_navigate,
    card_stack_update_viewport, card_stack_save_width,
)

from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS, ALIGN_CS_BTN_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_alignment.vad_card import (
    create_vad_card_renderer,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.models import AlignmentUrls
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.core import (
    _load_alignment_context, _update_alignment_state, _build_card_stack_state,
    _to_vad_chunks, _get_decomp_focused_index,
)

## Navigation Response Builder

In [ ]:
#| export
def _make_renderer(
    focused_segment_index:int=0,  # Currently focused text segment
) -> Any:  # Card renderer callback
    """Create a VAD card renderer with captured segment focus state."""
    return create_vad_card_renderer(
        focused_segment_index=focused_segment_index,
    )

def _build_nav_response(
    chunk_dicts:List[Dict[str, Any]],  # Serialized VAD chunks
    state:CardStackState,  # Current card stack state
    urls:AlignmentUrls,  # URL bundle
    focused_segment_index:int=0,  # Currently focused text segment
) -> Tuple:  # OOB response elements (slots + progress + focus)
    """Build OOB response for navigation changes."""
    chunks = _to_vad_chunks(chunk_dicts)
    return build_nav_response(
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=_make_renderer(focused_segment_index),
        progress_label="VAD Chunk",
    )

## Route Handlers

In [ ]:
#| export
def _handle_align_navigate(
    workflow:Any,  # StructureDecompWorkflow instance
    sess:Any,  # FastHTML session object
    direction:str,  # Navigation direction: up/down/first/last/page_up/page_down
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB response elements (slots + progress + focus)
    """Navigate the alignment card stack."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(workflow, session_id)
    chunks = _to_vad_chunks(ctx.chunk_dicts)

    state = _build_card_stack_state(ctx)
    focused_seg = _get_decomp_focused_index(workflow, session_id)
    renderer = _make_renderer(focused_seg)

    result = card_stack_navigate(
        direction=direction,
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
        progress_label="VAD Chunk",
    )

    _update_alignment_state(workflow, session_id, focused_chunk_index=state.focused_index)
    return result

In [ ]:
#| export
async def _handle_align_update_viewport(
    workflow:Any,  # StructureDecompWorkflow instance
    request:Any,  # FastHTML request object
    sess:Any,  # FastHTML session object
    visible_count:int,  # New visible card count
    urls:AlignmentUrls,  # URL bundle
) -> Tuple:  # OOB section elements for viewport update
    """Update viewport with new card count via OOB section swaps."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(workflow, session_id)
    chunks = _to_vad_chunks(ctx.chunk_dicts)

    state = _build_card_stack_state(ctx)
    focused_seg = _get_decomp_focused_index(workflow, session_id)
    renderer = _make_renderer(focused_seg)

    result = card_stack_update_viewport(
        visible_count=visible_count,
        card_items=chunks,
        state=state,
        config=ALIGN_CS_CONFIG,
        ids=ALIGN_CS_IDS,
        urls=urls.card_stack,
        render_card=renderer,
    )

    # Read is_auto from form data (passed by client JS)
    form_data = await request.form()
    is_auto_str = form_data.get("is_auto", "false")
    is_auto_mode = is_auto_str.lower() == "true"

    _update_alignment_state(
        workflow, session_id,
        visible_count=state.visible_count,
        is_auto_mode=is_auto_mode,
    )
    return result

In [ ]:
#| export
def _handle_align_save_width(
    workflow:Any,  # StructureDecompWorkflow instance
    sess:Any,  # FastHTML session object
    card_width:int,  # Card stack width in rem to save
) -> None:  # No response body (swap=none on client)
    """Save card stack width to server state with config bounds validation."""
    session_id = get_session_id(sess)
    ctx = _load_alignment_context(workflow, session_id)
    state = _build_card_stack_state(ctx)
    card_stack_save_width(state, card_width, ALIGN_CS_CONFIG)
    _update_alignment_state(workflow, session_id, card_width=state.card_width)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()